In [1]:
!pip install python.docx
!pip install docxedit

In [2]:
#PEGANDO O ARQUIVO
from docx import Document


documento = Document ("MODELO_PROCURACAO.docx")

for paragrafo in documento.paragraphs:
    paragrafo.text = paragrafo.text.replace("" , "")
    
    print  (paragrafo.text) 

PROCURAÇÃO

Por meio do presente instrumento particular, o OUTORGANTE: {OUTORGANTE}, nacionalidade {NACIONALIDADE}, profissão {PROFISSÃO}, portador da cédula de identidade RG n.º{RG},  e inscrito no CPF/MF sob o nº:  {CPF}, domiciliado à {DOMICILIADO}, nº  {N},  Bairro  {BAIRRO}, CEP  {CEP}, na cidade de {CIDADE}, nomeia e constitui seu bastante procurador o OUTORGADO: QUADRANTE INVESTIMENTOS LTDA, com sede na Cidade de São Paulo, Estado de São Paulo, na Av. Brigadeiro Faria Lima n.º 2277, 17° Andar – Ed. Plaza Iguatemi, CEP 01452-000, inscrita no CNPJ/MF sob o nº. 01.111.111/0001-12 (“Gestora”), devidamente autorizada pela Comissão de Valores Mobiliários – CVM ao exercício da administração profissional de carteiras de valores mobiliários, nos termos do Artigo 23 da Lei 6.385, de 07.12.1976, e da resolução CVM nº 21, de 25.02.2021, por seus representantes legais, EDSON SILVA, brasileiro, casado, empresário, portador da Cédula de identidade RG nº 10.001.011-1/SSP-SP e inscrito no CPF/MF

In [3]:
#EDITANDO OS DADOS
import os
import locale
from datetime import datetime

import pandas as pd
from docx import Document

# Isso vai fazer as datas saírem formatas com o nome em português
locale.setlocale(locale.LC_ALL, '')

# Coleta o nome do mês desejado para fazer a procuração
mes_desejado = input("Digite o mês desejado (Ex: janeiro, fevereiro, ...): ")

# Importa tabela do excel para o pandas
tabela = pd.read_excel("BANCO_DE_DADOS_CLT.xlsx")

# Esse método "fillna" completa todos os campos vazios com uma string vazia, isso é necessário para poder
# fazer o filtro no método seguinte. O argumento "inplace=True" faz com que a modificação seja feita no
# mesmo dataframe, sem precisar atribuir novamente a uma váriavel. Ex: tabela = tabela.fillna('')
tabela.fillna('', inplace=True)

# Filtra tabela com base no input recebido em "mes_desejado". O argumento "case=False" faz com que
# o filtro seja feito de forma "insentive", isto é, tratando letras maiúsculas e minúsculas como a mesma letra
# Ex: "OUTUBRO", "outubro" e "Outubro" vão significar a mesma coisa.
clientes_mes_desejado = tabela[tabela['validade'].str.contains(mes_desejado, case=False)]

# Aqui verificamos se não existe uma pasta chamada "docs" no mesmo diretório onde esse arquivo está
# Caso não exista, criamos a pasta. Serve para melhor organização
if not os.path.exists('Procurações'):
    os.mkdir('Procurações')

# Percorremos todas as linhas da tabela filtrada
for linha in clientes_mes_desejado.index:
    
    # Criamos a referência ao documento do word
    documento = Document("MODELO_PROCURACAO.docx")
    
    # Pegamos os valores de cada coluna usando a linha da iteração 
    # (ITERAÇÃO É QUANDO PERCORREMOS UM OBJETO EM UMA ESTRUTURA DE REPETIÇÃO IGUAL O for NO PYTHON)
    cod = tabela.loc[linha, "cod"]
    outorgante = tabela.loc[linha, "outorgante"]
    nacionalidade = tabela.loc[linha, "nacionalidade"]
    profissao = tabela.loc[linha, "profissão"]
    rg = tabela.loc[linha, "rg"]
    cpf = tabela.loc[linha, "cpf"]
    domiciliado = tabela.loc[linha, "domiciliado"]
    n = tabela.loc[linha, "n°"]
    bairro = tabela.loc[linha, "bairro"]
    cep = tabela.loc[linha, "cep"]
    cidade_estado = tabela.loc[linha, "cidade-estado"]
    validade = tabela.loc[linha, "validade"]
    data = tabela.loc[linha, "data"]
    nome = tabela.loc[linha, "nome"]
    
    # Associamos os valores a suas chaves que são usadas como placeholders no documento do word
    referencias = {
        "{OUTORGANTE}" : outorgante,
        "{NACIONALIDADE}" : nacionalidade,
        "{PROFISSÃO}" : profissao,
        "{RG}" : rg,
        "{CPF}" : cpf,
        "{DOMICILIADO}" : domiciliado,
        "{N}" : n,
        "{BAIRRO}" : bairro,
        "{CEP}" : cep,
        "{CIDADE}" : cidade_estado,
        "{VALIDADE}" : validade,
        "{DATA}" : datetime.now().strftime('%d de %B de %Y'),
        "{NOME}" : nome,
    }
    
    # Iteramos por cada parágrafo dentro do documento
    for paragrafo in documento.paragraphs:
        
        # Iteramos para cada run. Esse run seria pra cada conjunto de caracteres com a mesma formatação
        # Isso aqui é a lógica interna da lib que está sendo usada
        for run in paragrafo.runs:
            
            # Iteramos pelos itens do dicionário. Esse método "items()" do dicionário vai entregar uma
            # tupla contendo a chave como primeiro valor e o conteúdo da chave como segundo valor
            # Pra cada iteração é devolvido uma tupla como a seguinte:("{OUTORGANTE}", "Evelin Jeanette")
            # Aí em seguida é feito uma desestruturação
            for codigo, valor in referencias.items(): 
                
                # Substituimos os placeholders pelo valor. Passamos o valor dentro do método "str()" pra
                # garantir que todos os valores sejam uma string
                run.text = run.text.replace(codigo, str(valor))
    
    # Salvamos o documento dentro da pasta "docs" 
    documento.save(f"Procurações/{cod} - Procuração Qi.docx")
    
